# Course 9: Ten Years of Deep Learning Ideas
## Session 4: Vision transformers and the modern ConvNet

**Goal:** turn images into patch tokens, train a tiny vision transformer and a CNN with the same parameter budget at two training-set sizes, and measure what mixup and label smoothing change.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Setup and a synthetic image dataset
We draw 16x16 grayscale images of four shapes (disc, square, triangle, cross) at random positions, sizes, small rotations and brightness, with Gaussian noise. Each shape is drawn at 4 times the resolution and averaged down, so its edges are smooth. Real vision transformers use 224x224 colour images and millions of examples; our images are tiny so that each training run takes seconds.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(2)          # tiny models run as fast on 2 threads, and shared machines stay responsive
torch.set_flush_denormal(True)
torch.manual_seed(0); np.random.seed(0)
SIZE, SUPER = 16, 4
CLASSES = ["disc", "square", "triangle", "cross"]

def make_shapes(n, seed, noise=0.15):
    rng = np.random.default_rng(seed)
    c = (np.arange(SIZE * SUPER) + 0.5) / SUPER                 # centres of the fine pixels
    yy, xx = np.meshgrid(c, c, indexing="ij")
    labels = rng.integers(0, 4, n)
    imgs = np.zeros((n, SIZE, SIZE), np.float32)
    for i in range(n):
        r = rng.uniform(3.5, 6.5)
        cx, cy = rng.uniform(r, SIZE - r, 2)
        th = rng.uniform(-0.3, 0.3)                              # small rotation
        u = (xx - cx) * np.cos(th) + (yy - cy) * np.sin(th)
        v = -(xx - cx) * np.sin(th) + (yy - cy) * np.cos(th)
        k = labels[i]
        if k == 0:
            m = u ** 2 + v ** 2 <= r ** 2
        elif k == 1:
            m = (np.abs(u) <= 0.8 * r) & (np.abs(v) <= 0.8 * r)
        elif k == 2:
            m = np.ones_like(u, bool)
            for a in [0, 2 * np.pi / 3, 4 * np.pi / 3]:
                m &= (u * np.cos(a) + v * np.sin(a)) <= 0.5 * r
        else:
            m = ((np.abs(u) <= 0.3 * r) & (np.abs(v) <= r)) | ((np.abs(v) <= 0.3 * r) & (np.abs(u) <= r))
        imgs[i] = m.reshape(SIZE, SUPER, SIZE, SUPER).mean((1, 3))     # average down: smooth edges
    imgs = imgs * rng.uniform(0.5, 1.0, (n, 1, 1)) + rng.normal(0, noise, imgs.shape)
    return torch.tensor(imgs[:, None], dtype=torch.float32), torch.tensor(labels)

X_pool, y_pool = make_shapes(5000, seed=0)       # training pool
X_test, y_test = make_shapes(2000, seed=1)       # test set
print(f"training pool {tuple(X_pool.shape)}, test set {tuple(X_test.shape)}")

fig, ax = plt.subplots(1, 8, figsize=(9, 1.6))
for i in range(8):
    ax[i].imshow(X_pool[i, 0], cmap="gray"); ax[i].set_title(CLASSES[y_pool[i]], fontsize=8); ax[i].axis("off")
plt.suptitle("Examples of the synthetic shapes (16x16)", fontsize=9); plt.tight_layout(); plt.show()

### 2. An image as a sequence of tokens
A vision transformer (ViT; Dosovitskiy et al. 2021) cuts the image into patches, flattens each patch and maps it to a vector with one shared linear layer. A learned **class token** is put in front, and learned **position embeddings** are added, because attention by itself does not know where each patch was. Then come standard transformer blocks (Course 3 session 1 and Course 6 session 6 explain them), and a linear head reads the class token. With 4x4 patches, a 16x16 image becomes 16 patch tokens plus 1 class token. A convolution with kernel size = stride = patch size is exactly the 'flatten and project' step.

In [ ]:
patches = X_pool[:1].unfold(2, 4, 4).unfold(3, 4, 4)            # (1, 1, 4, 4, 4, 4): a 4x4 grid of 4x4 patches
print(f"one image -> {patches.shape[2] * patches.shape[3]} patches of {patches.shape[4] * patches.shape[5]} pixels each")


class Block(nn.Module):
    """Pre-norm transformer block. Attention is written out so that we can look at the attention weights later."""
    def __init__(self, d, heads, mlp_ratio=2):
        super().__init__()
        self.heads = heads
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, mlp_ratio * d), nn.GELU(), nn.Linear(mlp_ratio * d, d))
        self.last_attention = None

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)
        att = (q @ k.transpose(-2, -1) / math.sqrt(D // self.heads)).softmax(-1)
        self.last_attention = att.detach()
        x = x + self.proj((att @ v).transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))


class TinyViT(nn.Module):
    def __init__(self, d=64, depth=4, heads=4, patch=4, n_classes=4):
        super().__init__()
        n_patches = (SIZE // patch) ** 2
        self.patch_embed = nn.Conv2d(1, d, kernel_size=patch, stride=patch)      # flatten each patch + linear map
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n_patches + 1, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(depth)])
        self.norm, self.head = nn.LayerNorm(d), nn.Linear(d, n_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)                      # (B, 16, d)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], dim=1) + self.pos   # (B, 17, d)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x[:, 0]))                                     # read the class token

### 3. A small CNN with the same number of parameters
The CNN has what the ViT lacks: **locality** (each filter sees a 3x3 neighbourhood) and **translation equivariance** (the same filter slides over the whole image). These built-in assumptions are called inductive biases.

In [ ]:
def small_cnn(n_classes=4):
    def conv(cin, cout):
        return [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU()]
    return nn.Sequential(*conv(1, 16), nn.MaxPool2d(2), *conv(16, 32), *conv(32, 64), nn.MaxPool2d(2),
                         *conv(64, 192), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(192, n_classes))

count = lambda m: sum(p.numel() for p in m.parameters())
print(f"parameters: tiny ViT {count(TinyViT()):,}, small CNN {count(small_cnn()):,}")

### 4. Train both at two training-set sizes
Same recipe for both: AdamW (learning rate 2e-3, weight decay 0.05), 40 warmup steps, cosine decay, 400 steps of batch 128. With 500 images that is about 100 epochs; with 5,000 images about 10.

In [ ]:
def train(make_model, n_train, mix=False, steps=400, bs=128, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    X, y = X_pool[:n_train], y_pool[:n_train]
    g = torch.Generator().manual_seed(seed)
    beta_rng = np.random.default_rng(seed)
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = lr * min(1.0, (step + 1) / 40) * 0.5 * (1 + math.cos(math.pi * step / steps))
        idx = torch.randint(n_train, (bs,), generator=g)
        xb, yb = X[idx], y[idx]
        if mix:                                        # mixup (alpha 0.8) with label smoothing 0.1, as in DeiT
            lam = float(beta_rng.beta(0.8, 0.8))
            perm = torch.randperm(bs, generator=g)
            out = model(lam * xb + (1 - lam) * xb[perm])
            loss = (lam * F.cross_entropy(out, yb, label_smoothing=0.1)
                    + (1 - lam) * F.cross_entropy(out, yb[perm], label_smoothing=0.1))
        else:
            loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        test_acc = (model(X_test).argmax(1) == y_test).float().mean().item()
        train_acc = (model(X).argmax(1) == y).float().mean().item()
    return model, test_acc, train_acc

results, models = {}, {}
t0 = time.time()
for n in [500, 5000]:
    for name, maker in [("CNN", small_cnn), ("ViT", TinyViT)]:
        models[(name, n)], test_acc, train_acc = train(maker, n)
        results[(name, n)] = test_acc
        print(f"{name} trained on {n:4d} images: test accuracy {test_acc:.3f} (training accuracy {train_acc:.3f})")
print(f"(four runs took {time.time() - t0:.0f} s)")

### 5. The training recipe: mixup and label smoothing
DeiT (Touvron et al. 2021) showed that a ViT trained on ImageNet-1k alone can compete with CNNs if the recipe is strong: heavy augmentation (RandAugment, mixup, CutMix), label smoothing, stochastic depth and more. **Mixup** (Zhang et al. 2018) trains on blends of two images, λ·x₁ + (1 − λ)·x₂, with the same blend of their labels. **Label smoothing** (Szegedy et al. 2016) replaces the one-hot target by 1 − ε on the true class and ε/K spread over all K classes. We add both to the ViT trained on 500 images.

In [ ]:
_, acc_mix, train_mix = train(TinyViT, 500, mix=True)
results[("ViT + mixup + LS", 500)] = acc_mix
print(f"ViT on 500 images, plain recipe:            test accuracy {results[('ViT', 500)]:.3f}")
print(f"ViT on 500 images, mixup + label smoothing: test accuracy {acc_mix:.3f} (training accuracy {train_mix:.3f})")

names = ["CNN", "ViT", "ViT + mixup + LS"]
fig, ax = plt.subplots(figsize=(6, 3.5))
for j, n in enumerate([500, 5000]):
    vals = [results.get((nm, n), np.nan) for nm in names]
    ax.bar(np.arange(3) + 0.38 * j, vals, width=0.36, label=f"{n} training images")
ax.set_xticks(np.arange(3) + 0.19, names)
ax.set(ylim=(0.4, 1.0), ylabel="test accuracy", title="Same parameter budget, different inductive biases")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 6. What does the ViT attend to?
For one test image of each class, we show the attention from the class token to the 16 patches in the last block, averaged over the 4 heads (ViT trained on 5,000 images). Attention weights show where information is read from; they are not a complete explanation of the decision.

In [ ]:
vit = models[("ViT", 5000)]
idx = [int((y_test == k).nonzero()[0]) for k in range(4)]
with torch.no_grad():
    pred = vit(X_test[idx]).argmax(1)
att = vit.blocks[-1].last_attention[:, :, 0, 1:].mean(1).reshape(-1, 4, 4)    # class token -> 16 patches

fig, ax = plt.subplots(2, 4, figsize=(9, 3.5))
for j in range(4):
    ax[0, j].imshow(X_test[idx[j], 0], cmap="gray")
    ax[0, j].set_title(f"{CLASSES[int(y_test[idx[j]])]} (pred: {CLASSES[int(pred[j])]})", fontsize=8)
    ax[1, j].imshow(att[j], cmap="viridis")
    ax[1, j].set_title("class-token attention", fontsize=8)
    for a in ax[:, j]:
        a.axis("off")
plt.tight_layout(); plt.show()

### 7. A real ViT (NOT RUN here)
The `timm` library has DeiT models pre-trained on ImageNet. The cell below fine-tunes DeiT-tiny (5.7M parameters, 16x16 patches on 224x224 images) on CIFAR-10. It follows the documented `timm` and `torchvision` interfaces; check them before running.

In [ ]:
# NOT RUN: needs `pip install timm torchvision`, a download of CIFAR-10 and the DeiT weights, and a GPU.
import timm, torch, torchvision
model = timm.create_model("deit_tiny_patch16_224", pretrained=True, num_classes=10).cuda()
config = timm.data.resolve_data_config({}, model=model)
train_tf = timm.data.create_transform(**config, is_training=True)
test_tf = timm.data.create_transform(**config)
train_set = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=train_tf)
test_set = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=test_tf)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, num_workers=2)
opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
for epoch in range(2):
    model.train()
    for x, y in train_loader:
        loss = torch.nn.functional.cross_entropy(model(x.cuda()), y.cuda(), label_smoothing=0.1)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (model(x.cuda()).argmax(1).cpu() == y).sum().item()
    print(f"epoch {epoch}: CIFAR-10 test accuracy {correct / len(test_set):.3f}")
# Then repeat with timm.create_model("resnet18", pretrained=True, num_classes=10) and compare.

### TODO
1. Train the ViT with mixup and label smoothing on 5,000 images. Does the recipe still help at the larger size with this short schedule? Then double the number of steps and compare again.
2. Change the patch size to 2 (64 patch tokens). What happens to accuracy and to the time per step?
3. Complete the token counter below.

In [ ]:
# TODO: number of tokens a ViT sees for an image of side `image` with square patches of side `patch` (include the class token).
def n_tokens(image, patch):
    return None          # replace: (image // patch) ** 2 + 1

t = n_tokens(224, 16)
print("not done yet" if t is None else f"224x224 with 16x16 patches: {t} tokens (expected 197)")

### What to notice
WRITTEN AFTER THE RUN